# Dựng artefact offline trên Colab và Kaggle

Notebook này dùng GPU Colab T4 hoặc GPU Kaggle để dựng index ảnh, index văn bản và caption từ artefact KB/ảnh có sẵn. Các cell chỉ gọi CLI của `artrag`; không tự triển khai logic nghiệp vụ. Có thể lưu artefact đầu vào trên Google Drive hoặc trong một Kaggle Dataset. `REPO_URL` trong cell cấu hình đã trỏ tới repository `https://github.com/HoangDucHiep/art_rag.git`; nếu nhân bản sang repo khác thì hãy thay lại URL cho khớp.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/HoangDucHiep/art_rag.git"
REPO_DIR = "/kaggle/working/art_rag" if Path("/kaggle/working").exists() else "/content/art_rag"
PROFILE = "kaggle" if Path("/kaggle/input").exists() else "colab"
ARTIFACTS_NAME = "artifacts.zip"
BUILD_LIMIT_ARGS = ""  # Ví dụ chạy thử: "--limit 100"; để trống để dựng đầy đủ.

os.environ["PROFILE"] = PROFILE
os.environ["BUILD_LIMIT_ARGS"] = BUILD_LIMIT_ARGS
os.environ["REPO_DIR"] = REPO_DIR
os.environ["ARTIFACTS_NAME"] = ARTIFACTS_NAME

print(f"Profile: {PROFILE} | Repo: {REPO_DIR}")

In [ ]:
from datetime import datetime
import shlex
import shutil
import subprocess
import sys
import time

repo_path = Path(REPO_DIR)
if (repo_path / ".git").is_dir():
    print(f"Repository đã có sẵn: {repo_path}")
elif repo_path.exists():
    raise RuntimeError(f"Thư mục repo đã tồn tại nhưng không phải Git repository: {repo_path}")
else:
    if REPO_URL == "https://github.com/OWNER/REPOSITORY.git":
        raise ValueError("Hãy thay REPO_URL bằng URL Git thật của repository trước khi chạy cell này.")
    repo_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    if not (repo_path / ".git").is_dir():
        raise RuntimeError(f"Clone hoàn tất nhưng không tìm thấy Git repository tại {repo_path}")

os.chdir(repo_path)
Path("results").mkdir(parents=True, exist_ok=True)
with Path("results/offline_build.md").open("a", encoding="utf-8") as report:
    report.write(f"\n## Lần chạy {datetime.now().astimezone().isoformat(timespec='seconds')} ({PROFILE})\n")

def run_step(name, cmd):
    started = time.perf_counter()
    completed = subprocess.run(cmd, check=False)
    elapsed = time.perf_counter() - started
    status = "OK" if completed.returncode == 0 else f"FAILED (exit {completed.returncode})"
    with Path("results/offline_build.md").open("a", encoding="utf-8") as report:
        report.write(f"- {name}: {elapsed:.2f} seconds ({status})\n")
    if completed.returncode:
        raise subprocess.CalledProcessError(completed.returncode, cmd)
    return completed

run_step("pip install -e .", [sys.executable, "-m", "pip", "install", "-e", "."])

`pip install -e .` là cách cài đơn giản trên Colab/Kaggle; notebook giữ nguyên ghim `transformers>=4.46,<4.47`. Nếu pip gặp lỗi giải quyết `torch`, có thể chạy `pip install -e . --no-deps` rồi cài thủ công các dependency còn thiếu, dùng bản torch tương thích với runtime hiện tại.

In [ ]:
if not Path("/kaggle/input").exists():
    from google.colab import drive

    drive.mount("/content/drive")
else:
    print("Kaggle input datasets are mounted at /kaggle/input.")

In [ ]:
Path("data").mkdir(parents=True, exist_ok=True)
archive_path = Path(ARTIFACTS_NAME)
if archive_path.is_file():
    print(f"Đã có artefact tại {archive_path.resolve()}.")
else:
    archive_source = None
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        archive_source = next(kaggle_input.rglob(ARTIFACTS_NAME), None)
    drive_archive = Path("/content/drive/MyDrive") / ARTIFACTS_NAME
    if archive_source is None and drive_archive.is_file():
        archive_source = drive_archive

    if archive_source is not None:
        shutil.copy2(archive_source, archive_path)
        print(f"Artefact copied to {repo_path / archive_path}")
    else:
        print("Không tìm thấy artifacts.zip. Hãy tải artefact lên Drive (Colab) hoặc thêm dataset chứa artifacts.zip (Kaggle), rồi chạy lại cell này. Nếu chưa có artefact, cần fetch dữ liệu CMA, build KB và tải ảnh trước; các bước mạng dài này không tự chạy trong notebook.")

## Giải nén dữ liệu

Nếu cell tìm artefact báo không có `artifacts.zip`, hãy cung cấp file đó trước khi tiếp tục. Tái tạo từ nguồn cần fetch dữ liệu CMA, build KB (bao gồm dữ liệu họa sĩ nếu cần) và tải ảnh bằng các module `artrag.data`; notebook không tự khởi chạy các tác vụ mạng dài này.

In [ ]:
run_step("unpack", [sys.executable, "-m", "artrag.tools.artifacts", "unpack", ARTIFACTS_NAME, "--dest", "data", "--strict"])

In [ ]:
run_step("clip_index build", [sys.executable, "-m", "artrag.index.clip_index", "build", "--profile", PROFILE, *shlex.split(BUILD_LIMIT_ARGS)])

In [ ]:
run_step("text_index build", [sys.executable, "-m", "artrag.index.text_index", "build", "--profile", PROFILE, *shlex.split(BUILD_LIMIT_ARGS)])

In [ ]:
run_step("captions build", [sys.executable, "-m", "artrag.index.captions", "build", "--profile", PROFILE, *shlex.split(BUILD_LIMIT_ARGS)])

In [ ]:
run_step("clip_index check", [sys.executable, "-m", "artrag.index.clip_index", "check", "--profile", PROFILE])
run_step("text_index check", [sys.executable, "-m", "artrag.index.text_index", "check", "--profile", PROFILE])

Cả hai lệnh kiểm tra cần in trạng thái `OK`; đối chiếu `kb_hash index` với `kb_hash hiện tại` để xác nhận index dùng đúng KB.

In [ ]:
run_step("clip_index search", [sys.executable, "-m", "artrag.index.clip_index", "search", "two men rowing a boat", "--top", "5", "--profile", PROFILE])

In [ ]:
run_step("artifacts pack", [sys.executable, "-m", "artrag.tools.artifacts", "pack", "--profile", PROFILE, "--only", "kb,images,index,cache", "--out", ARTIFACTS_NAME])
print(f"{archive_path.stat().st_size:,} bytes: {archive_path.resolve()}")
drive_dir = Path("/content/drive/MyDrive")
if drive_dir.is_dir():
    shutil.copy2(archive_path, drive_dir / ARTIFACTS_NAME)
    print("Đã lưu artifacts.zip lên Google Drive.")
else:
    print("Trên Kaggle, artifacts.zip nằm trong /kaggle/working; hãy lưu output thành Kaggle Dataset.")

Thời gian từng lệnh được đo bằng `time.perf_counter()` quanh `subprocess.run` trong tiện ích `run_step` và ghi vào `results/offline_build.md`. Cách này không phụ thuộc `/usr/bin/time` có sẵn trên Colab/Kaggle; lệnh lỗi cũng được ghi trạng thái trước khi lỗi được báo lại. Sau khi chạy xong, `artifacts.zip` được lưu lên Google Drive trong Colab; trên Kaggle, tạo hoặc cập nhật Kaggle Dataset từ file trong `/kaggle/working`.